# 03 – SQL basics on the portfolio database

Core SQL on my own data: `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY`, `HAVING` and `JOIN`. The database is built by notebook 02 (`data/processed/portfolio.db`).

Each section shows a worked query, followed by an exercise. Write the exercise yourself first; the solutions are at the bottom.

## Setup

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

In [2]:
DB_PATH = Path("../data/processed/portfolio.db")
assert DB_PATH.exists(), "Run notebook 02 first"


def query(sql):
    """Run a SQL query on the portfolio database and return a DataFrame."""
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql(sql, con)

In [3]:
query("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,holdings
1,symbols
2,prices_local
3,fx_ecb
4,returns_eur


## 1. SELECT and LIMIT

`SELECT` chooses columns, `FROM` the table, `LIMIT` the number of rows. SQL keywords in capitals is a convention, not a rule.

In [4]:
query("""
SELECT ticker, name, pie, weight_in_portfolio
FROM holdings
LIMIT 5
""")

,ticker,name,pie,weight_in_portfolio
0,V,Visa,Kwaliteit,0.0189
1,MCO,Moody's,Kwaliteit,0.0189
2,LLY,Eli Lilly,Kwaliteit,0.0189
3,RMS,Hermès International,Kwaliteit,0.0189
4,REL,RELX,Kwaliteit,0.0162


## 2. WHERE: filtering rows

`WHERE` keeps rows that meet a condition. Combine conditions with `AND` / `OR`; text goes between single quotes.

In [5]:
query("""
SELECT ticker, name, weight_in_portfolio
FROM holdings
WHERE pie = 'Energie' AND weight_in_portfolio > 0.015
""")

,ticker,name,weight_in_portfolio
0,WENS,iShares MSCI World Energy Sector (Dist),0.0486
1,LNG,Cheniere Energy,0.0180
2,LIN,Linde,0.0162
3,CEG,Constellation Energy,0.0162


## 3. ORDER BY: sorting

`ORDER BY column DESC` sorts from large to small. Combined with `LIMIT` this gives a top-n.

In [19]:
query("""
SELECT ticker, pie, weight_in_portfolio
FROM holdings
ORDER BY weight_in_portfolio DESC
LIMIT 10
""")

,ticker,pie,weight_in_portfolio
0,GDX,Mijnbouw,0.0544
1,WENS,Energie,0.0486
2,EPIN,Multipolaire Wereldorde,0.0375
3,LOCK,Thematisch,0.0364
4,GRID,Thematisch,0.0351
5,WQTM,Thematisch,0.0286
6,AMEL,Multipolaire Wereldorde,0.0270
7,SILV,Mijnbouw,0.0255
8,CHN,Multipolaire Wereldorde,0.0225
9,IUSS,Multipolaire Wereldorde,0.0225


## 4. GROUP BY and HAVING: aggregating

`GROUP BY` makes one row per group; `SUM`, `COUNT`, `AVG`, `MIN`, `MAX` summarise each group. `WHERE` filters rows **before** grouping, `HAVING` filters groups **after** grouping. This is the SQL version of `groupby` in pandas.

In [25]:
query("""
SELECT pie, block, COUNT(*) AS n_positions, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY pie, block
ORDER BY block, weight DESC
""")

,pie,block,n_positions,weight
0,Kwaliteit,A,17,0.27
1,Thematisch,A,5,0.13
2,AI Boom,A,8,0.10
3,Energie,B,10,0.18
4,Mijnbouw,B,9,0.17
5,Multipolaire Wereldorde,B,6,0.15


In [8]:
query("""
SELECT country, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY country
HAVING SUM(weight_in_portfolio) > 0.05
ORDER BY weight DESC
""")

,country,weight
0,Ierland,0.3871
1,VS,0.2945
2,Canada,0.0916
3,Zwitserland,0.0594


## 5. JOIN: combining tables

`JOIN ... ON` links rows of two tables on a shared column, exactly like `merge` in pandas. Short table aliases (`h`, `s`) keep the query readable. `LEFT JOIN` keeps every row of the left table, even without a match.

In [9]:
query("""
SELECT h.ticker, h.pie, h.weight_in_portfolio, s.yahoo_symbol, s.series_type, s.proxy_symbol
FROM holdings AS h
JOIN symbols AS s ON h.ticker = s.ticker
WHERE s.series_type = 'alt_listing' OR s.proxy_symbol IS NOT NULL
ORDER BY h.weight_in_portfolio DESC
""")

,ticker,pie,weight_in_portfolio,yahoo_symbol,series_type,proxy_symbol
0,WENS,Energie,0.0486,WENS.L,listing,WNRG.L
1,EPIN,Multipolaire Wereldorde,0.0375,EPI.L,alt_listing,EPI
2,GRID,Thematisch,0.0351,GRDU.L,alt_listing,GRID
3,WQTM,Thematisch,0.0286,WQTM.L,alt_listing,NaN
4,AMEL,Multipolaire Wereldorde,0.0270,ALAT.PA,alt_listing,NaN
5,SILV,Mijnbouw,0.0255,SILV.MI,listing,SIL
6,CHN,Multipolaire Wereldorde,0.0225,ICHN.AS,alt_listing,NaN
7,IUSS,Multipolaire Wereldorde,0.0225,IKSA.L,alt_listing,NaN
8,LYXI,Multipolaire Wereldorde,0.0225,INDO.PA,alt_listing,NaN
9,VGEK,Multipolaire Wereldorde,0.0180,VAPU.L,alt_listing,NaN


## 6. Putting it together on the returns table

How many trading days per ticker come from a proxy, and from which date does each ticker have data? `MIN`, `MAX` and a conditional count with `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`.

In [10]:
query("""
SELECT ticker,
       MIN(date) AS first_date,
       COUNT(*) AS n_days,
       SUM(CASE WHEN source = 'proxy' THEN 1 ELSE 0 END) AS proxy_days
FROM returns_eur
GROUP BY ticker
HAVING proxy_days > 0 OR MIN(date) > '2019-02-01'
ORDER BY first_date
""")

,ticker,first_date,n_days,proxy_days
0,EPIN,2019-01-03 00:00:00,2024,1847
1,GRID,2019-01-03 00:00:00,2024,850
2,SILV,2019-01-03 00:00:00,2024,878
3,WENS,2019-01-03 00:00:00,2024,904
4,IUSS,2019-04-11 00:00:00,1954,0
5,CHN,2019-06-21 00:00:00,1904,0
6,VGEK,2019-09-25 00:00:00,1836,0
7,DAGB,2021-05-07 00:00:00,1419,0
8,COPX,2021-11-23 00:00:00,1277,0
9,CEG,2022-01-20 00:00:00,1235,0


The worst days of the **current** portfolio (a backtest with today's weights, so it contains hindsight). A JOIN with the weights, a weighted sum per date, sorted ascending. `SUM(h.weight_in_portfolio)` on the same rows shows the coverage of each day: positions without data do not count.

In [11]:
query("""
SELECT r.date,
       SUM(r.return_total * h.weight_in_portfolio) AS portfolio_return,
       SUM(h.weight_in_portfolio) AS coverage
FROM returns_eur AS r
JOIN holdings AS h ON r.ticker = h.ticker
GROUP BY r.date
ORDER BY portfolio_return ASC
LIMIT 5
""")

,date,portfolio_return,coverage
0,2020-03-12 00:00:00,-0.084360,0.9120
1,2020-03-09 00:00:00,-0.083811,0.9120
2,2020-03-16 00:00:00,-0.073022,0.9120
3,2020-03-18 00:00:00,-0.060061,0.9120
4,2025-04-04 00:00:00,-0.058012,0.9714


## Solutions

Try each exercise first. Compare afterwards: there is often more than one correct query.

In [12]:
# Solution 1
query("""
SELECT ticker, name, exchange, listing_currency
FROM holdings
LIMIT 10
""")

,ticker,name,exchange,listing_currency
0,V,Visa,NYSE,USD
1,MCO,Moody's,NYSE,USD
2,LLY,Eli Lilly,NYSE,USD
3,RMS,Hermès International,Euronext Parijs,EUR
4,REL,RELX,LSE,GBP
5,DHR,Danaher,NYSE,USD
6,IDXX,IDEXX Laboratories,NASDAQ,USD
7,ISRG,Intuitive Surgical,NASDAQ,USD
8,GIVN,Givaudan,SIX,CHF
9,SIKA,Sika,SIX,CHF


In [13]:
# Solution 2
query("""
SELECT ticker, name, exchange, listing_currency
FROM holdings
WHERE asset_type = 'ETF' AND listing_currency = 'EUR'
""")

,ticker,name,exchange,listing_currency
0,GRID,First Trust Nasdaq Clean Edge Smart Grid Infra...,Xetra,EUR
1,WQTM,WisdomTree Quantum Computing,Xetra,EUR
2,GDX,VanEck Gold Miners UCITS,Borsa Italiana,EUR
3,SILV,Global X Silver Miners UCITS,Borsa Italiana,EUR
4,EPIN,WisdomTree India Earnings,Xetra,EUR
5,AMEL,Amundi MSCI EM Latin America,Xetra,EUR
6,CHN,iShares MSCI China (Acc),Xetra,EUR
7,IUSS,iShares MSCI Saudi Arabia Capped,Xetra,EUR
8,LYXI,Amundi MSCI Indonesia,Xetra,EUR
9,VGEK,Vanguard FTSE Developed Asia Pacific ex Japan,Xetra,EUR


In [14]:
# Solution 3
query("""
SELECT ticker, name, dividend_yield
FROM holdings
ORDER BY dividend_yield DESC
LIMIT 5
""")

,ticker,name,dividend_yield
0,RIO,Rio Tinto,0.0572
1,ENB,Enbridge,0.0557
2,WKL,Wolters Kluwer,0.0299
3,EOG,EOG Resources,0.0246
4,REL,RELX,0.0230


In [15]:
# Solution 4
query("""
SELECT listing_currency, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY listing_currency
ORDER BY weight DESC
""")

,listing_currency,weight
0,USD,0.4403
1,EUR,0.3795
2,GBP,0.1089
3,CHF,0.0459
4,CAD,0.0254


In [16]:
# Solution 5
query("""
SELECT s.series_type, COUNT(*) AS n_positions, SUM(h.weight_in_portfolio) AS weight
FROM holdings AS h
JOIN symbols AS s ON h.ticker = s.ticker
GROUP BY s.series_type
""")

,series_type,n_positions,weight
0,alt_listing,10,0.2434
1,listing,45,0.7566


In [17]:
# Solution 6
query("""
SELECT r.date,
       SUM(r.return_total * h.weight_in_portfolio) AS portfolio_return,
       SUM(h.weight_in_portfolio) AS coverage
FROM returns_eur AS r
JOIN holdings AS h ON r.ticker = h.ticker
WHERE r.date >= '2022-01-01'
GROUP BY r.date
ORDER BY portfolio_return DESC
LIMIT 5
""")

,date,portfolio_return,coverage
0,2022-11-10 00:00:00,0.048714,0.9451
1,2022-05-13 00:00:00,0.028881,0.9451
2,2025-04-09 00:00:00,0.027375,0.9714
3,2026-04-08 00:00:00,0.025597,1.0000
4,2023-01-06 00:00:00,0.025500,0.9451


## Practice

In [26]:
query("""
SELECT ticker, name, country
FROM holdings
WHERE country = 'Zwitserland'
""")

,ticker,name,country
0,GIVN,Givaudan,Zwitserland
1,SIKA,Sika,Zwitserland
2,CB,Chubb,Zwitserland
3,STMN,Straumann,Zwitserland


In [27]:
query("""
SELECT ticker, name, weight_in_portfolio
FROM holdings
WHERE pie = 'Energie' AND weight_in_portfolio > 0.015
""")

,ticker,name,weight_in_portfolio
0,WENS,iShares MSCI World Energy Sector (Dist),0.0486
1,LNG,Cheniere Energy,0.0180
2,LIN,Linde,0.0162
3,CEG,Constellation Energy,0.0162


In [28]:
query("""
SELECT ticker, name, weight_in_portfolio
FROM holdings
WHERE pie = 'Mijnbouw'
""")

,ticker,name,weight_in_portfolio
0,GDX,VanEck Gold Miners UCITS,0.0544
1,SILV,Global X Silver Miners UCITS,0.0255
2,RIO,Rio Tinto,0.0153
3,FNV,Franco-Nevada,0.0136
4,FCX,Freeport-McMoRan,0.0136
5,WPM,Wheaton Precious Metals,0.0119
6,HBM,Hudbay Minerals,0.0119
7,WREE,WisdomTree Strategic Metals & Rare Earths Miners,0.0119
8,NXE,NexGen Energy,0.0119


In [29]:
query("""
SELECT ticker, name, exchange
FROM holdings
WHERE asset_type = 'ETF' AND listing_currency = 'EUR'
""")

,ticker,name,exchange
0,GRID,First Trust Nasdaq Clean Edge Smart Grid Infra...,Xetra
1,WQTM,WisdomTree Quantum Computing,Xetra
2,GDX,VanEck Gold Miners UCITS,Borsa Italiana
3,SILV,Global X Silver Miners UCITS,Borsa Italiana
4,EPIN,WisdomTree India Earnings,Xetra
5,AMEL,Amundi MSCI EM Latin America,Xetra
6,CHN,iShares MSCI China (Acc),Xetra
7,IUSS,iShares MSCI Saudi Arabia Capped,Xetra
8,LYXI,Amundi MSCI Indonesia,Xetra
9,VGEK,Vanguard FTSE Developed Asia Pacific ex Japan,Xetra


In [30]:
query("""
SELECT ticker, country
FROM holdings
WHERE country = 'Canada' OR country = 'VS'
""")

,ticker,country
0,V,VS
1,MCO,VS
2,LLY,VS
3,DHR,VS
4,IDXX,VS
5,ISRG,VS
6,CSU,Canada
7,EW,VS
8,SYK,VS
9,NVDA,VS


In [31]:
query("""
SELECT ticker, name, ROUND(weight_in_portfolio * 100, 2) AS weight_pct
FROM holdings
WHERE pie = 'Energie'
ORDER BY weight_pct DESC
LIMIT 3
""")

,ticker,name,weight_pct
0,WENS,iShares MSCI World Energy Sector (Dist),4.86
1,LNG,Cheniere Energy,1.80
2,LIN,Linde,1.62


In [32]:
query("""
SELECT ticker, name, dividend_yield
FROM holdings
ORDER BY dividend_yield DESC
LIMIT 5
""")

,ticker,name,dividend_yield
0,RIO,Rio Tinto,0.0572
1,ENB,Enbridge,0.0557
2,WKL,Wolters Kluwer,0.0299
3,EOG,EOG Resources,0.0246
4,REL,RELX,0.0230


In [33]:
query("""
SELECT ticker, weight_in_portfolio
FROM holdings
ORDER BY weight_in_portfolio ASC
LIMIT 3
""")

,ticker,weight_in_portfolio
0,VRT,0.0100
1,AI,0.0108
2,CDNS,0.0110


In [35]:
query("""
SELECT ticker, ROUND(weight_in_portfolio * 100, 1) AS weight_pct
FROM holdings
WHERE pie = 'AI Boom'
ORDER BY weight_pct DESC
""")

,ticker,weight_pct
0,NVDA,1.5
1,TSM,1.4
2,MSFT,1.3
3,ASML,1.3
4,GOOGL,1.2
5,AVGO,1.2
6,CDNS,1.1
7,VRT,1.0


In [36]:
query("""
SELECT pie, COUNT(*) AS n_positions, ROUND(AVG(weight_in_portfolio) * 100, 2) AS avg_weight_pct
FROM holdings
GROUP BY pie
ORDER BY avg_weight_pct DESC
""")

,pie,n_positions,avg_weight_pct
0,Thematisch,5,2.60
1,Multipolaire Wereldorde,6,2.50
2,Mijnbouw,9,1.89
3,Energie,10,1.80
4,Kwaliteit,17,1.59
5,AI Boom,8,1.25


In [38]:
query("""
SELECT pie, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY pie
HAVING SUM(weight_in_portfolio) >= 0.15
""")

,pie,weight
0,Energie,0.18
1,Kwaliteit,0.27
2,Mijnbouw,0.17
3,Multipolaire Wereldorde,0.15


In [39]:
query("""
SELECT block, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY block
""")

,block,weight
0,A,0.5
1,B,0.5


In [41]:
query("""
SELECT asset_type, COUNT(*) AS n_positions, SUM(weight_in_portfolio) * 100 AS weight_pct
FROM holdings
GROUP BY asset_type
ORDER BY weight_pct DESC
""")

,asset_type,n_positions,weight_pct
0,Aandeel,40,57.96
1,ETF,15,42.04


In [46]:
query("""
SELECT country, COUNT(*) AS n_positions
FROM holdings
GROUP BY country
HAVING COUNT(*) >= 3
ORDER BY COUNT(*) DESC
""")

,country,n_positions
0,VS,20
1,Ierland,14
2,Canada,7
3,Zwitserland,4


In [47]:
query("""
SELECT h.ticker, h.name, h.exchange, s.yahoo_symbol, s.series_type
FROM holdings AS h
JOIN symbols AS s ON h.ticker = s.ticker
WHERE h.pie = 'Kwaliteit'
""")

,ticker,name,exchange,yahoo_symbol,series_type
0,V,Visa,NYSE,V,listing
1,MCO,Moody's,NYSE,MCO,listing
2,LLY,Eli Lilly,NYSE,LLY,listing
3,RMS,Hermès International,Euronext Parijs,RMS.PA,listing
4,REL,RELX,LSE,REL.L,listing
5,DHR,Danaher,NYSE,DHR,listing
6,IDXX,IDEXX Laboratories,NASDAQ,IDXX,listing
7,ISRG,Intuitive Surgical,NASDAQ,ISRG,listing
8,GIVN,Givaudan,SIX,GIVN.SW,listing
9,SIKA,Sika,SIX,SIKA.SW,listing


In [48]:
query("""
SELECT h.ticker
FROM holdings AS h
LEFT JOIN symbols AS s ON h.ticker = s.ticker
WHERE s.ticker IS NULL
""")

,ticker


In [57]:
query("""
SELECT s.series_type, COUNT(*) AS n_positions, SUM(h.weight_in_portfolio) AS weight
FROM symbols AS s
JOIN holdings AS h ON s.ticker = h.ticker
GROUP BY s.series_type 
""")

,series_type,n_positions,weight
0,alt_listing,10,0.2434
1,listing,45,0.7566


In [63]:
query("""
SELECT h.pie,
       SUM(CASE WHEN source = 'proxy' THEN 1 ELSE 0 END) AS proxy_days
FROM returns_eur AS r
JOIN holdings AS h ON r.ticker = h.ticker
WHERE r.source = 'proxy'
GROUP BY h.pie
""")

,pie,proxy_days
0,Energie,904
1,Mijnbouw,878
2,Multipolaire Wereldorde,1847
3,Thematisch,850


In [64]:
query("""
SELECT r.ticker, COUNT(*) AS proxy_days, MIN(r.date) AS first_proxy_day, MAX(r.date) AS last_proxy_day
FROM returns_eur AS r
WHERE r.source = 'proxy'
GROUP BY r.ticker
ORDER BY proxy_days DESC
""")

,ticker,proxy_days,first_proxy_day,last_proxy_day
0,EPIN,1847,2019-01-03 00:00:00,2026-02-24 00:00:00
1,WENS,904,2019-01-03 00:00:00,2022-06-30 00:00:00
2,SILV,878,2019-01-03 00:00:00,2022-05-25 00:00:00
3,GRID,850,2019-01-03 00:00:00,2022-04-14 00:00:00
